# Организации с более чем 10 предписаниями в Redis
Исходный файл: https://github.com/junaart/ForStudents/raw/refs/heads/main/Big_Data/data_%D0%A3%D0%9A.xlsx. Суммы штрафов указаны в тысячах рублей.

In [ ]:
import pandas as pd
import redis
import uuid

url = (
    "https://github.com/junaart/ForStudents/raw/refs/heads/main/Big_Data/"
    "data_%D0%A3%D0%9A.xlsx"
)
org_col = "Наименование управляющей организации, жилищного объединения"
count_col = "Выдано предписаний"
fine_col = "Сумма штрафа (тыс. руб.)"

db = redis.Redis(host="localhost", port=6379, db=0, decode_responses=True)
db.ping()
df = pd.read_excel(url, engine="openpyxl")
selected = df.loc[df[count_col] > 10, [org_col, count_col, fine_col]].copy()
selected[org_col] = selected[org_col].str.strip()
assert selected[org_col].notna().all() and selected[org_col].is_unique
print("Всего строк:", len(df))
print("Организаций с предписаниями > 10:", len(selected))

In [ ]:
hashes = {}
for name, count, fine in selected.itertuples(index=False, name=None):
    key = "uk:org:" + str(uuid.uuid5(uuid.NAMESPACE_URL, name))
    fields = {
        "Организация": name,
        "Выдано предписаний": int(count),
        "Сумма штрафов": str(fine),
    }
    db.hset(key, mapping=fields)
    hashes[key] = {field: str(value) for field, value in fields.items()}

keys_in_redis = set(db.scan_iter(match="uk:org:*"))
assert keys_in_redis == set(hashes)
assert all(db.type(key) == "hash" for key in hashes)
assert all(db.hgetall(key) == hashes[key] for key in hashes)
print("Хэш-таблиц в Redis:", len(hashes))

In [ ]:
sample_key = max(hashes, key=lambda key: int(hashes[key][count_col]))
print("Ключ:", sample_key)
print("Тип Redis:", db.type(sample_key))
print("Содержимое хэша:", db.hgetall(sample_key))